In [1]:
import os
os.chdir(r"E:\text_summarizer")
print(os.getcwd())

E:\text_summarizer


In [2]:
from dataclasses import dataclass
from pathlib import Path
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [3]:
@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: str

In [4]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path = CONFIG_FILE_PATH,
        params_file_path = PARAMS_FILE_PATH):
    
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        
        create_directories([self.config.artifacts_root])
        
        
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        
        create_directories([config.root_dir])
        
        data_transformation_config = DataTransformationConfig(
            root_dir = config.root_dir,
            data_path = config.data_path,
            tokenizer_name = config.tokenizer_name
        )
        
        return data_transformation_config     

In [5]:
import os
from textSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

e:\text_summarizer\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)
        
    def convert_example_to_features(self, example_batch):

        inputs = example_batch["dialogue"]
        targets = example_batch["summary"]

        # Ensure all elements are strings
        inputs = [str(x) for x in inputs]
        targets = [str(x) for x in targets]

        model_inputs = self.tokenizer(
            inputs,
            max_length=512,
            padding="max_length",
            truncation=True
        )

        labels = self.tokenizer(
            targets,
            max_length=64,
            padding="max_length",
            truncation=True
        )

        model_inputs["labels"] = labels["input_ids"]

        return model_inputs

    
    
    def convert(self):
        dataset = load_dataset(
            "csv",
            data_files={
                "train": "artifacts/data_ingestion/samsum-train.csv",
                "test": "artifacts/data_ingestion/samsum-test.csv",
                "validation": "artifacts/data_ingestion/samsum-validation.csv"
            }
        )

        dataset_pt = dataset.map(self.convert_example_to_features, batched=True)

        dataset_pt.save_to_disk(
            os.path.join(self.config.root_dir, "samsum_dataset")
        )

In [7]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e

[2026-02-18 08:35:50,334: INFO: textSummarizer: YAML file: E:\text_summarizer\config\config.yaml loaded successfully]
[2026-02-18 08:35:50,339: INFO: textSummarizer: YAML file: E:\text_summarizer\params.yaml loaded successfully]
[2026-02-18 08:35:50,342: INFO: textSummarizer: created directory at: artifacts]
[2026-02-18 08:35:50,342: INFO: textSummarizer: created directory at: artifacts/data_transformation]
[2026-02-18 08:35:50,646: INFO: httpx: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/config.json "HTTP/1.1 200 OK"]
[2026-02-18 08:35:50,893: INFO: httpx: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"]
[2026-02-18 08:35:51,151: INFO: httpx: HTTP Request: GET https://huggingface.co/api/models/t5-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"]
[2026-02-18 08:35:51,385: INFO: httpx: HTTP Request: GET https://huggingface.co/api/models/google-t5/t5-small/t

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 82383.44 examples/s] 
